# **Benchmarking**
In this notebook, we will be comparing our algorithm to other state of art algorithms

## **1. Imports and Benchmarking**

### **1.1 Imports**

In [ ]:
import torch
import numpy as np
import torchvision
import matplotlib.pyplot as plt
import kagglehub
from transformers import AutoImageProcessor, AutoModel
import os

### **1.2 Model Prep**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
from huggingface_hub import login
login()

In [ ]:
model_name = "facebook/dinov3-vitl16-pretrain-lvd1689m"
#model_name = "facebook/dinov3-vit7b16-pretrain-lvd1689m" # Flagship

print(f"Loading flagship model: {model_name}")
print(f"Targeting Architecture: 40 Layers, 4096 Embedding Dim")

if 'dtype' not in locals():
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32

try:
    #Load Model with Memory Optimization and Eager Attention
    # We strictly need eager attention to capture the attention weights
    model = AutoModel.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True,
        attn_implementation="eager"
    ).to(device).eval()

    #Load Processor
    processor = AutoImageProcessor.from_pretrained(model_name)

    num_registers = getattr(model.config, "num_register_tokens", 4)
    print(f"Model loaded. Number of register tokens identified: {num_registers}")

    transform = lambda img: processor(images=img, return_tensors="pt")['pixel_values'].squeeze()

    #Verify Layer Count
    if hasattr(model, "vision_model"):
        num_layers = len(model.vision_model.encoder.layers)
    else:
        #Fallback for flat architectures
        num_layers = len(model.layer)

    print(f"Success! Model loaded with {num_layers} layers.")

except Exception as e:
    print(f"Error loading model: {e}")
    print("Tip: If you run out of memory (OOM), try 'dinov3-vitl16' (Large, 24 layers) instead.")

# **2. Dataset Download**

In [ ]:
os.makedirs('./data', exist_ok=True)

### **2.1 ID**

In [ ]:
cifar10_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
cifar10_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
print(f"✅ CIFAR-10 Loaded: {len(cifar10_train)} Train | {len(cifar10_test)} Test")

In [ ]:
cifar100_train = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
cifar100_test = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)
print(f"✅ CIFAR-100 Loaded: {len(cifar100_train)} Train | {len(cifar100_test)} Test")

### **2.2 OOD**

In [ ]:
svhn_test = torchvision.datasets.SVHN(root='./data', split='test', download=True, transform=transform)
print(f"✅ SVHN (OOD) Loaded: {len(svhn_test)} Test images")

In [ ]:
dtd_test = torchvision.datasets.DTD(root='./data', split='test', download=True, transform=transform)
print(f"✅ DTD Textures (OOD) Loaded: {len(dtd_test)} Test images")

In [ ]:
places365_test = torchvision.datasets.Places365(
        root='./data',
        split='val',
        small=True,
        download=True,
        transform=transform
    )
print(f"✅ Places365 (OOD) Loaded: {len(places365_test)} images")

In [ ]:
os.makedirs('./data/lsun', exist_ok=True)
    lsun_test = torchvision.datasets.LSUN(
        root='./data/lsun',
        classes='test',
        transform=transform
    )
print(f"✅ LSUN (OOD) Loaded: {len(lsun_test)} images")

### **2.3 Dataloaders**

In [ ]:
BATCH_SIZE = 16

In [ ]:
id_train_loader = DataLoader(cifar100_train, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
id_test_loader = DataLoader(cifar100_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
svhn_loader = DataLoader(svhn_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
dtd_loader = DataLoader(dtd_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
places_loader = DataLoader(places365_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

### **2.4 Synthetic Noise**

In [ ]:
class SyntheticNoiseDataset(Dataset):
    def __init__(self, size=10000, img_size=(32, 32, 3), noise_type='gaussian', transform=None):
        """
        Generates 10,000 synthetic noise images to match standard benchmark sizes.
        """
        self.size = size
        self.img_size = img_size
        self.noise_type = noise_type
        self.transform = transform

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        if self.noise_type == 'gaussian':
            # Mean 128, Std 64, clipped to 0-255
            arr = np.random.normal(128, 64, self.img_size)
            arr = np.clip(arr, 0, 255).astype(np.uint8)
        elif self.noise_type == 'uniform':
            # Uniformly distributed between 0-255
            arr = np.random.uniform(0, 255, self.img_size).astype(np.uint8)

        img = Image.fromarray(arr)

        if self.transform:
            img = self.transform(img)

        return img, -1 # -1 label signifies OOD

### **2.5 Synthetic Noise**

In [ ]:
gaussian_dataset = SyntheticNoiseDataset(noise_type='gaussian', transform=transform)
uniform_dataset = SyntheticNoiseDataset(noise_type='uniform', transform=transform)

gaussian_loader = DataLoader(gaussian_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
uniform_loader = DataLoader(uniform_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## **3. Feature Extraction**

In [ ]:
os.makedirs('./features', exist_ok=True)

### **3.1 Function definition**

In [ ]:
@torch.no_grad()
def extract_and_save_features(model, dataloader, device, save_path):
    if os.path.exists(save_path):
        print(f"⏭️ Skipping {save_path} (Already exists)")
        return

    model.eval()
    all_features = []
    all_labels = []

    for images, labels in tqdm(dataloader, desc=f"Extracting -> {os.path.basename(save_path)}"):
        images = images.to(device)

        # Forward pass
        outputs = model(pixel_values=images)

        # Extract the [CLS] token (global image representation)
        cls_tokens = outputs.last_hidden_state[:, 0, :].cpu()

        all_features.append(cls_tokens)
        all_labels.append(labels)

    all_features = torch.cat(all_features, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    torch.save({'features': all_features, 'labels': all_labels}, save_path)
    print(f"✅ Saved {len(all_features)} features to {save_path}")

### **3.2 ID Extraction**

In [ ]:
extract_and_save_features(model, id_train_loader, device, './features/id_cifar100_train.pt')
extract_and_save_features(model, id_test_loader, device, './features/id_cifar100_test.pt')

### **3.3 OOD Extraction**

In [ ]:
extract_and_save_features(model, svhn_loader, device, './features/ood_svhn.pt')
extract_and_save_features(model, dtd_loader, device, './features/ood_dtd.pt')
extract_and_save_features(model, places_loader, device, './features/ood_places.pt')
extract_and_save_features(model, gaussian_loader, device, './features/ood_gaussian.pt')
extract_and_save_features(model, uniform_loader, device, './features/ood_uniform.pt')